# Lab 07: Prediction System / Inference Pipeline

**Goal:** Send new retail information through the fitted model pipeline and return a sales prediction, without training the model again.

## Project phase

```text
Phase 1 data understanding -> Phase 2 EDA -> Phase 3 feature engineering
-> Phase 4 baseline -> Phase 5 evaluation -> Phase 6 model improvement
-> Phase 7 prediction system (here) -> Phase 8 dashboard integration -> Phase 9 deployment
```

## Inference architecture

```text
Retail input -> Pydantic validation -> feature preparation -> saved preprocessing + model pipeline
            -> predicted sales -> JSON response -> frontend
```

The React frontend already exists in `frontend/`; this lab verifies the backend contract it can call. We inspect the saved model's real input features and exercise the same request schema and predictor used by FastAPI.

In [32]:
from pathlib import Path
import sys
import joblib

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from backend.predictor import MODEL_FEATURES, MODEL_PATH
from backend.schemas import PredictionRequest
from sklearn.pipeline import Pipeline

assert MODEL_PATH == PROJECT_DIR / 'models' / 'sales_prediction_model.joblib'
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Saved model pipeline not found at {MODEL_PATH}')

model = joblib.load(MODEL_PATH)
if not isinstance(model, Pipeline):
    raise TypeError(f'Expected a saved sklearn Pipeline, got {type(model).__name__}')

print(f'Model path: {MODEL_PATH}')
print(f'Model type: {type(model).__name__}')
print(f'Pipeline steps: {list(model.named_steps)}')
print(f'Model input features ({len(MODEL_FEATURES)}): {MODEL_FEATURES}')


Model path: E:\NIAI\Projects\Retail Sales Prediction\models\sales_prediction_model.joblib
Model type: Pipeline
Pipeline steps: ['preprocessor', 'model']
Model input features (12): ['Store', 'DayOfWeek', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend']


## 7.1 Training versus inference

Training uses historical rows with `model.fit(X_train, y_train)` to learn parameters. Inference uses the already-fitted artifact with `model.predict(new_data)`; it does **not** retrain for each request.

```text
Training:  historical data -> fit -> fitted pipeline saved to disk
Inference: new request   -> saved pipeline.predict() -> sales estimate
```

Phase 6 contains candidate comparisons, but its notebook has no saved execution results establishing a winner. This app therefore uses the existing saved 12-feature pipeline that the API is configured to load; we do not claim this lab re-ran that model comparison.

In [33]:
from backend.predictor import predict_sales, prepare_feature_row

sample = {
    'store': 1,
    'dayOfWeek': 5,
    'date': '2015-07-31',
    'customers': 555,
    'promo': True,
    'stateHoliday': '0',
    'schoolHoliday': True,
}
request = PredictionRequest(**sample)
feature_row = prepare_feature_row(request)

assert list(feature_row.columns) == MODEL_FEATURES
prediction = predict_sales(request)
print(f'Prepared one row with {len(feature_row.columns)} model features.')
print(f'Predicted sales: {prediction:,.2f} dataset units')


Prepared one row with 12 model features.
Predicted sales: 5,304.38 dataset units


## 7.2 The selected artifact and why the pipeline matters

This repository has two distinct model artifacts: `sales_linear_regression.pkl` is a plain 9-feature baseline; `sales_prediction_model.joblib` is the 12-feature fitted `Pipeline` used by the production API. They are not interchangeable. The request used here is retail-friendly (store, date, customers, promotions, holidays); the backend derives calendar features and supplies the columns the pipeline was fitted with.

Always deploy the **whole fitted preprocessing + estimator pipeline**, not just its final regressor. The pipeline applies the same categorical encoding and numeric preprocessing used during training.

After selecting and fitting the final candidate in model development, persist that complete object once, for example:

```python
joblib.dump(final_model, PROJECT_DIR / 'models' / 'sales_prediction_model.joblib')
```

This inference lab verifies the already-saved artifact rather than retraining or overwriting it.

In [34]:
# The JSON fields match PredictionRequest and the frontend PredictionInput.
import json
print(json.dumps(sample, indent=2))


{
  "store": 1,
  "dayOfWeek": 5,
  "date": "2015-07-31",
  "customers": 555,
  "promo": true,
  "stateHoliday": "0",
  "schoolHoliday": true
}


## 7.3 Feature preparation and DataFrame input

The API accepts business inputs rather than requiring a caller to compute engineered fields. `prepare_feature_row` derives `Year`, `Month`, `Day`, `WeekOfYear`, and `IsWeekend`, and creates a one-row DataFrame with the pipeline's exact, named feature columns. A DataFrame helps prevent accidentally swapping positional feature values.

## 7.4 Request schema and input validation

`backend/schemas.py` uses Pydantic to validate the fields before inference: store must be positive, weekday must be 1-7, customers cannot be negative, and `open` must be 0 or 1. The date is parsed in feature preparation. Invalid requests should receive a clear client error instead of reaching the model.

In [35]:
if list(feature_row.columns) != list(model.feature_names_in_):
    raise ValueError('Prepared request features do not match the features used to fit the saved model.')

print('Pipeline steps:', list(model.named_steps))
print('Model input features:', list(model.feature_names_in_))
print('Prepared feature row:')
display(feature_row)

Pipeline steps: ['preprocessor', 'model']
Model input features: ['Store', 'DayOfWeek', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend']
Prepared feature row:


,Store,DayOfWeek,Customers,Open,Promo,StateHoliday,SchoolHoliday,Year,Month,Day,WeekOfYear,IsWeekend
0,1,5,555,1,1,0,1,2015,7,31,31,0


In [36]:
FINAL_MODEL_PATH = MODEL_PATH
if not FINAL_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Saved model pipeline not found at {FINAL_MODEL_PATH}')

print(f'Complete fitted pipeline exists at: {FINAL_MODEL_PATH}')

Complete fitted pipeline exists at: E:\NIAI\Projects\Retail Sales Prediction\models\sales_prediction_model.joblib


In [37]:
loaded_model = joblib.load(FINAL_MODEL_PATH)
if not isinstance(loaded_model, Pipeline):
    raise TypeError('Reloaded artifact is not a preprocessing + model Pipeline.')

print(f'Reloaded model type: {type(loaded_model).__name__}')

Reloaded model type: Pipeline


In [38]:
print(loaded_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['DayOfWeek', 'Customers',
                                                   'Open', 'Promo',
                                                   'SchoolHoliday', 'Year',
                                                   'Month', 'Day', 'WeekOfYear',
                                                   'IsWeekend']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Store', 'StateHoliday'])])),
                ('model', LinearRegression())])


In [39]:
test_prediction = float(loaded_model.predict(feature_row)[0])
assert abs(test_prediction - prediction) < 1e-8
print(f'Prediction before save/load: {prediction:.8f}')
print(f'Prediction after save/load:  {test_prediction:.8f}')
print('Save/load prediction check passed.')

Prediction before save/load: 5304.37697105
Prediction after save/load:  5304.37697105
Save/load prediction check passed.


## 7.5 Run and test the FastAPI service

From the project root, start the service in a terminal:

```powershell
uvicorn backend.main:app --reload
```

Open `http://127.0.0.1:8000/docs` and try `POST /api/predict` with the sample JSON above. The API also exposes `POST /predict` and `GET /health`. The frontend uses the `/api/predict` route.

The response includes `predictedSales`, `lowerBound`, `upperBound`, `modelStatus`, and the accepted input. Sales are returned in the dataset's units; do not label the result with a currency unless the source data's currency is confirmed. The following cell calls the same route handler in-process; Swagger UI is the manual HTTP check.

In [40]:
from backend.main import health, predict as api_predict, root
from fastapi import HTTPException
from pydantic import ValidationError

print('Root:', await root())
print('Health:', await health())

api_result = await api_predict(request)
response_json = api_result.model_dump()
assert response_json['modelStatus'] == 'trained'
assert isinstance(response_json['predictedSales'], float)
assert abs(response_json['predictedSales'] - round(prediction, 2)) < 1e-8
print('Prediction endpoint response:')
print(json.dumps(response_json, indent=2))

test_inputs = [
    sample,
    {**sample, 'store': 25, 'dayOfWeek': 6, 'date': '2015-08-01', 'customers': 1200, 'promo': False, 'stateHoliday': 'a', 'schoolHoliday': False},
    {**sample, 'store': 1115, 'dayOfWeek': 7, 'date': '2015-08-02', 'customers': 0, 'promo': False, 'schoolHoliday': False, 'open': 0},
]
for test_input in test_inputs:
    test_request = PredictionRequest(**test_input)
    test_result = await api_predict(test_request)
    print(f"Store {test_request.store}, {test_request.date}: {test_result.predictedSales:,.2f} dataset units")

try:
    PredictionRequest(store=0, dayOfWeek=8, date='2015-07-31', customers=-1)
except ValidationError as exc:
    print(f'Invalid request rejected with {len(exc.errors())} field validation errors.')
else:
    raise AssertionError('Invalid request unexpectedly passed validation.')

try:
    await api_predict(PredictionRequest(store=1, dayOfWeek=5, date='not-a-date', customers=100))
except HTTPException as exc:
    assert exc.status_code == 400
    print(f'Invalid date rejected with HTTP {exc.status_code}: {exc.detail}')
else:
    raise AssertionError('Invalid date unexpectedly reached model inference.')


Root: {'message': 'Retail Sales Prediction API is running', 'status': 'online', 'model': 'sales_prediction_model.joblib'}
Health: {'status': 'ok', 'model_loaded': True, 'model_path': 'E:\\NIAI\\Projects\\Retail Sales Prediction\\models\\sales_prediction_model.joblib', 'pipeline_type': 'Pipeline'}
Prediction endpoint response:
{
  "predictedSales": 5304.38,
  "lowerBound": 4349.59,
  "upperBound": 6259.16,
  "modelStatus": "trained",
  "input": {
    "store": 1,
    "dayOfWeek": 5,
    "date": "2015-07-31",
    "customers": 555,
    "promo": true,
    "stateHoliday": "0",
    "schoolHoliday": true
  }
}
Store 1, 2015-07-31: 5,304.38 dataset units
Store 25, 2015-08-01: 8,041.16 dataset units
Store 1115, 2015-08-02: 2,179.84 dataset units
Invalid request rejected with 3 field validation errors.
Invalid date rejected with HTTP 400: date must be a valid ISO date string (YYYY-MM-DD).


## Phase 7 checklist

- [x] Load the existing complete preprocessing + model pipeline.
- [x] Inspect model input columns and verify prepared input order and names.
- [x] Validate retail request fields before inference.
- [x] Derive calendar features and predict with the saved pipeline.
- [x] Confirm predictions are unchanged by save/load.
- [x] Exercise health and prediction response handlers with several requests.
- [x] Confirm invalid fields and dates are rejected.
- [ ] Start Uvicorn and test the endpoint interactively at `/docs`.

The final flow is: JSON input -> Pydantic schema -> feature preparation -> saved pipeline -> prediction JSON. The dashboard can consume this response shape in the integration phase.